# Tabela comparativa de métricas

Estrutura:
1. **Carrega** os dados brutos de cada dataset em `df_bioflore` e `df_amazon` (colunas padronizadas)
2. **Visualiza** os DataFrames
3. **Gera tabelas comparativas** a partir dos DataFrames

## Imports e configuração

In [ ]:
import yaml
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "bioflore_data").exists() else Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

In [ ]:
# /home/luizluz/Documentos/multi-task-fcn/bioflore_data/vLaura_v3

In [ ]:
# ── Versões "Our Model" ── altere AQUI para trocar o modelo em todo o notebook ─
BIO_OUR_MODEL = "v02"
AMZ_OUR_MODEL = "13_amazon_data_v3"

# Formato: (label_exibição, pasta, average_all_iters)
# average_all_iters=True  → nosso modelo: média de todas as iterações nas tabelas
# average_all_iters=False → concorrente:  iter_001 apenas nas tabelas
BIOFLORE_VERSIONS = [
    ("Our Model",  BIO_OUR_MODEL,        True),
    ("DeepLabV3", "vDeepLabv3Vanilla",  False),
    ("Multi task DeepLabv3",      "vLaura_v4",         False),
]
AMAZON_VERSIONS = [
    ("Our Model",  AMZ_OUR_MODEL,                          True),
    ("DeepLabV3", "vDeepLabv3Vanilla_amazon_v512_run_01",        False),
    ("Multi task DeepLabv3",      "vLaura_deeplabvlaura_resnet50_v4", False),
]



## 1. Carregamento dos dados brutos

Cada dataset é carregado em um DataFrame com colunas **padronizadas** (`pixel_*` e `comp_*`).  
As chaves dos YAMLs são mapeadas explicitamente aqui — fácil de ajustar se os nomes mudarem.

In [ ]:
# ── BIOFLORE ─────────────────────────────────────────────────────────────────
# global_component_metrics.yaml → métricas componente/ITC (nível de copa)

rows = []
for label, folder, _ in BIOFLORE_VERSIONS:
    version_path = REPO_ROOT / "bioflore_data" / folder
    with open(version_path / "args.yaml") as f:
        num_iter = yaml.safe_load(f).get("num_iter", 0)

    for i in range(1, num_iter + 1):
        row = {"version": label, "iter": i}

        c = version_path / f"iter_{i:03d}" / "global_component_metrics.yaml"
        if c.exists():
            with open(c) as f:
                m = yaml.safe_load(f)
            row["comp_accuracy"] = m.get("global/component/Accuracy")
            row["comp_f1"]       = m.get("global/component/avgF1")
            row["comp_prec"]     = m.get("global/component/avgPrec")
            row["comp_rec"]      = m.get("global/component/avgRec")

        rows.append(row)

df_bioflore = pd.DataFrame(rows)
print(f"df_bioflore: {df_bioflore.shape}")
display(df_bioflore.head(20))



In [ ]:
# ── AMAZON ────────────────────────────────────────────────────────────────────
# all_labels_test_metrics.yaml → métricas componente/ITC (nível de copa)

rows = []
for label, folder, _ in AMAZON_VERSIONS:
    version_path = REPO_ROOT / "amazon_data" / folder
    with open(version_path / "args.yaml") as f:
        num_iter = yaml.safe_load(f).get("num_iter", 0)

    for i in range(1, num_iter + 1):
        row = {"version": label, "iter": i}

        c = version_path / f"iter_{i:03d}" / "all_labels_test_metrics.yaml"
        if c.exists():
            with open(c) as f:
                m = yaml.safe_load(f)
            row["comp_f1"]       = m["all_labels_avgF1_component"]
            row["comp_prec"]     = m["all_labels_avgPrec_component"]
            row["comp_rec"]      = m["all_labels_avgRec_component"]
            row["comp_accuracy"] = m["all_labels_Accuracy_component"]

        rows.append(row)

df_amazon = pd.DataFrame(rows)
print(f"df_amazon: {df_amazon.shape}")
display(df_amazon.head(20))

## 2. Tabelas comparativas

`comparison_table(df, coluna)` agrega os dados:
- **Our Model** → média de todas as iterações
- **Concorrentes** → valor da iter. 001

Troque o argumento `coluna` por qualquer coluna do DataFrame para comparar outra métrica.

In [ ]:
def comparison_table(df: pd.DataFrame, coluna: str, our_model_label: str = "Our Model") -> pd.DataFrame:
    """Agrega df em uma tabela comparativa de uma coluna de métrica."""
    rows = []
    for version in df["version"].unique():
        vdf = df[df["version"] == version]
        if version == our_model_label:
            val  = vdf[coluna].mean()
            nota = "avg. todas as iter."
        else:
            iter1 = vdf[vdf["iter"] == 1][coluna].values
            val   = iter1[0] if len(iter1) > 0 else None
            nota  = "iter. 001"
        rows.append({"Modelo": version, coluna: round(val, 2) if val is not None and not pd.isna(val) else None, "Nota": nota})
    return pd.DataFrame(rows).set_index("Modelo")

In [ ]:
# Comparação mostrando o resultado de uma iteração específica para cada modelo

def iter_comparison_table(df: pd.DataFrame, colunas, iter_num: int, our_model_label: str = "Our Model") -> pd.DataFrame:
    """Agrega df em uma tabela comparativa mostrando o resultado de uma iteração específica (iter_num).
    colunas: str ou lista de str com as métricas a exibir.
    """
    if isinstance(colunas, str):
        colunas = [colunas]
    rows = []
    for version in df["version"].unique():
        vdf = df[df["version"] == version]
        if version == our_model_label:
            subset = vdf[vdf["iter"] == iter_num]
            nota = f"iter. {iter_num:03d}"
        else:
            subset = vdf[vdf["iter"] == 1]
            nota = "iter. 001"
        row = {"Modelo": version, "Nota": nota}
        for col in colunas:
            vals = subset[col].values
            row[col] = round(vals[0], 2) if len(vals) > 0 and not pd.isna(vals[0]) else None
        rows.append(row)
    return pd.DataFrame(rows).set_index("Modelo")

# Exemplo de uso: mostrar a comp_f1 da iteração 20 para "Our Model"
bioflore_selected_iter = 18
amazon_selected_iter = 13

for df, dataset_label, sel_iter in [
    (df_bioflore, "Bioflore Maranhão", bioflore_selected_iter),
    (df_amazon,   "Embrapa Amazon",    amazon_selected_iter),
]:
    for col, label in [("comp_f1", "F1"), ("comp_prec", "Precision"), ("comp_rec", "Recall")]:
        print(f"=== {dataset_label} — Component-level {label} (iteração {sel_iter}) ===")
        display(iter_comparison_table(df, col, sel_iter))

### Gráfico de barras — Component-level F1 (ITC)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_fig_dir = REPO_ROOT / "exploration_notebooks" / "figures" / "compare_models_metrics"
_fig_dir.mkdir(parents=True, exist_ok=True)

def plot_comp_f1_bar(tbl, title, filename):
    """Gráfico de barras do comp_f1 para um dataset."""
    models = tbl["Modelo"].tolist()
    values = tbl["comp_f1"].tolist()
    x = np.arange(len(models))
    COLORS = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0"]
    colors = [COLORS[i % len(COLORS)] for i in range(len(models))]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(x, values, color=colors, alpha=0.88, width=0.5)

    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(
                bar.get_x() + bar.get_width() / 2, h + 0.4,
                f"{h:.1f}", ha="center", va="bottom", fontsize=10,
            )

    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=11)
    ax.set_ylabel("Component-level F1-score (%)", fontsize=12)
    ax.set_ylim(0, 110)
    ax.yaxis.set_tick_params(labelsize=11)
    ax.grid(axis="y", alpha=0.25, linewidth=0.7, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(_fig_dir / f"{filename}.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(_fig_dir / f"{filename}.png", dpi=300, bbox_inches="tight")
    print(f"Saved: {_fig_dir / filename}.pdf")
    plt.show()

tbl_bio = iter_comparison_table(df_bioflore, "comp_f1", bioflore_selected_iter).reset_index()
tbl_amz = iter_comparison_table(df_amazon,   "comp_f1", amazon_selected_iter).reset_index()

plot_comp_f1_bar(tbl_bio, "Bioflore MA — Component-level F1", "comp_f1_bar_bioflore")
plot_comp_f1_bar(tbl_amz, "Amazon AC — Component-level F1",   "comp_f1_bar_amazon")


## 3. Evolução por iteração — N° de ITCs e F1-score (Our Model)

4 linhas: N° de ITCs (azul) e F1-score componente (laranja), com estilos de linha distintos por dataset (sólido = Bioflore, tracejado = Amazon).

In [ ]:
import yaml
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "bioflore_data").exists() else Path.cwd().parent

# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   12,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "axes.linewidth":   0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

# ── Colorblind-friendly palette (Wong 2011) ──────────────────────────────────
COLOR_ITC = "#0072B2"   # blue      → N° de ITCs
COLOR_F1  = "#D55E00"   # vermilion → F1-score
LW        = 2.5         # line width
MS        = 6           # marker size (Bioflore only)

# ── 1. Load N° of ITCs per iteration ─────────────────────────────────────────
def load_itc_counts(folder, parquet_name):
    counts = []
    for i in range(1, 50):
        p = folder / f"iter_{i:03d}" / parquet_name
        if not p.exists():
            break
        df = pd.read_parquet(p)
        counts.append((i, len(df)))
    return counts

bio_folder = REPO_ROOT / "bioflore_data" / BIO_OUR_MODEL
amz_folder = REPO_ROOT / "amazon_data"   / AMZ_OUR_MODEL

bio_itcs = load_itc_counts(bio_folder, "all_regions_stats.parquet")
amz_itcs = load_itc_counts(amz_folder, "all_labels_stats.parquet")

# ── 2. Load F1-score per iteration ───────────────────────────────────────────
def load_f1_bioflore(folder):
    rows = []
    for i in range(1, 50):
        c = folder / f"iter_{i:03d}" / "global_component_metrics.yaml"
        if not c.exists():
            break
        with open(c) as f:
            m = yaml.safe_load(f)
        f1 = m.get("global/component/avgF1")
        if f1 is not None:
            rows.append((i, f1))
    return rows

def load_f1_amazon(folder):
    rows = []
    for i in range(1, 50):
        c = folder / f"iter_{i:03d}" / "all_labels_test_metrics.yaml"
        if not c.exists():
            break
        with open(c) as f:
            m = yaml.safe_load(f)
        f1 = m["all_labels_avgF1_component"]
        rows.append((i, f1))
    return rows

bio_f1 = load_f1_bioflore(bio_folder)
amz_f1 = load_f1_amazon(amz_folder)

# ── 3. Plot ───────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax2 = ax1.twinx()

xs_bio = [x[0] for x in bio_itcs]
xs_amz = [x[0] for x in amz_itcs]

# N° de ITCs — Bioflore: solid + circle markers; Amazon: dashed, no markers
ax1.plot(xs_bio, [x[1] for x in bio_itcs],
         color=COLOR_ITC, linestyle="-",  linewidth=LW,
         marker="o", markersize=MS, markevery=2,
         label="Nº ITCs – Bioflore MA")
ax1.plot(xs_amz, [x[1] for x in amz_itcs],
         color=COLOR_ITC, linestyle="--", linewidth=LW,
         label="Nº ITCs – Amazon AC")

# F1-score — Bioflore: solid + circle markers; Amazon: dashed, no markers
ax2.plot([x[0] for x in bio_f1], [x[1] for x in bio_f1],
         color=COLOR_F1, linestyle="-",  linewidth=LW,
         marker="o", markersize=MS, markevery=2,
         label="F1-score – Bioflore MA")
ax2.plot([x[0] for x in amz_f1], [x[1] for x in amz_f1],
         color=COLOR_F1, linestyle="--", linewidth=LW,
         label="F1-score – Amazon AC")

# ── Axes formatting ───────────────────────────────────────────────────────────
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Nº of ITCs", color=COLOR_ITC)
ax1.tick_params(axis="y", labelcolor=COLOR_ITC)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax1.set_xlim(left=1)
ax1.set_ylim(bottom=0)

ax2.set_ylabel("F1 Score", color=COLOR_F1)
ax2.tick_params(axis="y", labelcolor=COLOR_F1)
ax2.set_ylim(0, 100)

# Subtle grid on primary axis only
ax1.grid(True, linestyle=":", linewidth=0.6, color="grey", alpha=0.5)
ax1.set_axisbelow(True)

# Remove top & right spines for a clean publication look
for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)

# Combined legend (bottom-right, clean frame)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="lower right", frameon=True,
           framealpha=0.95, edgecolor="lightgrey")

fig.tight_layout()
_fig_dir = REPO_ROOT / "exploration_notebooks" / "figures" / "compare_models_metrics"
_fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(_fig_dir / "itc_f1_evolution.pdf", bbox_inches="tight", dpi=300)
plt.savefig(_fig_dir / "itc_f1_evolution.png", bbox_inches="tight", dpi=300)
plt.show()
print("Saved.")

## 4. Mean distance of crown area from species median — Our Model

Two lines: one per dataset (Bioflore MA and Amazon AC), **Our Model only**.  
For each crown the distance is `|area_m² – median_area_m²_of_its_species|`.  
Species medians are computed from the training reference of each dataset (iter\_000 for Amazon; iter\_001 for Bioflore, which has no iter\_000).

In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import rasterio
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "bioflore_data").exists() else Path.cwd().parent

# ── Style (consistent with the rest of the notebook) ─────────────────────────
mpl.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   12,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "axes.linewidth":   0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

# Colorblind-friendly palette (Wong 2011)
COLOR_BIO = "#0072B2"   # blue      → Bioflore MA
COLOR_AMZ = "#D55E00"   # vermilion → Amazon AC
LW        = 2.5

# ── Pixel scale for Amazon (m/pixel) ─────────────────────────────────────────
_ortho_amz = REPO_ROOT / "amazon_data" / "amazon_input_data" / "orthoimage" / "NOV_2017_FINAL_004.tif"
with rasterio.open(_ortho_amz) as _src:
    AMZ_PIXEL_M = abs(_src.transform.a)   # ≈ 0.04 m/pixel

# Bioflore: area column is already in m² (confirmed from parquet values ~38 m²)
BIO_PIXEL_M = 1.0   # no conversion needed

# ── Helper: compute species medians from a reference parquet ─────────────────
def species_medians(parquet_path: Path, pixel_m: float) -> pd.Series:
    df = pd.read_parquet(parquet_path)
    df["area_m2"] = df["area"] * pixel_m ** 2
    return df.groupby("tree_type")["area_m2"].median()

# Amazon: no iter_000 stats → use iter_001 as reference
amz_medians = species_medians(
    REPO_ROOT / "amazon_data" / AMZ_OUR_MODEL / "iter_001" / "all_labels_stats.parquet",
    AMZ_PIXEL_M,
)
# Bioflore: no iter_000 → use iter_001 as reference
bio_medians = species_medians(
    REPO_ROOT / "bioflore_data" / BIO_OUR_MODEL / "iter_001" / "all_regions_stats.parquet",
    BIO_PIXEL_M,
)

# ── Helper: load mean |area_m2 - species_median| per iteration ───────────────
def load_mean_area_distance(
    version_folder: Path,
    parquet_name: str,
    pixel_m: float,
    medians: pd.Series,
    start_iter: int = 1,
) -> pd.DataFrame:
    rows = []
    for i in range(start_iter, 50):
        p = version_folder / f"iter_{i:03d}" / parquet_name
        if not p.exists():
            break
        df = pd.read_parquet(p)
        df["area_m2"] = df["area"] * pixel_m ** 2
        df = df.merge(medians.rename("median_m2"), on="tree_type", how="left")
        df["dist_m2"] = (df["area_m2"] - df["median_m2"]).abs()
        rows.append({"iter_num": i, "mean_dist_m2": df["dist_m2"].mean()})
    return pd.DataFrame(rows)

# ── Load Our Model for both datasets ─────────────────────────────────────────
df_bio = load_mean_area_distance(
    REPO_ROOT / "bioflore_data" / BIO_OUR_MODEL,
    parquet_name="all_regions_stats.parquet",
    pixel_m=BIO_PIXEL_M,
    medians=bio_medians,
    start_iter=2,   # iter_001 is the reference, start from iter_002
)
df_amz = load_mean_area_distance(
    REPO_ROOT / "amazon_data" / AMZ_OUR_MODEL,
    parquet_name="all_labels_stats.parquet",
    pixel_m=AMZ_PIXEL_M,
    medians=amz_medians,
    start_iter=1,
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(df_bio["iter_num"], df_bio["mean_dist_m2"],
        color=COLOR_BIO, linestyle="-", linewidth=LW,
        marker="o", markersize=5, markevery=2,
        label="Our Model – Bioflore MA")
ax.plot(df_amz["iter_num"], df_amz["mean_dist_m2"],
        color=COLOR_AMZ, linestyle="--", linewidth=LW,
        label="Our Model – Amazon AC")

ax.set_xlabel("Iteration")
ax.set_ylabel("Area (m²)")
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax.set_xlim(left=1)
ax.grid(True, linestyle=":", linewidth=0.6, color="grey", alpha=0.5)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2,
          frameon=True, framealpha=0.95, edgecolor="lightgrey")

fig.tight_layout()
_fig_dir = REPO_ROOT / "exploration_notebooks" / "figures" / "compare_models_metrics"
_fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(_fig_dir / "mean_area_distance_from_species_median.pdf", bbox_inches="tight", dpi=300)
plt.savefig(_fig_dir / "mean_area_distance_from_species_median.png", bbox_inches="tight", dpi=300)
plt.show()
print("Saved.")

## 4. Comparativo por espécie — F1 a nível de componente (ITC)

Dois gráficos independentes, um por dataset.  
- **Eixo X**: espécie  
- **Cada grupo de barras**: um método (Our Model / DeepLabV3 / Multi task DeepLabv3)  
- **Métrica**: F1-score a nível de componente  
- **Our Model**: média de todas as iterações (± desvio-padrão nas barras de erro)  
- **Concorrentes**: iteração 001

> **Nota Amazon**: Os arquivos de métricas do *Our Model* (13_amazon_data) não armazenam  
> F1 por espécie a nível de componente, portanto ele não aparece no gráfico da Amazon.

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Paleta Okabe-Ito (daltonismo-segura) ─────────────────────────────────────
COLORS = {
    "Our Model":  "#0072B2",   # azul
    "DeepLabV3": "#E69F00",   # laranja
    "Multi task DeepLabv3":      "#009E73",   # verde
}

# ── Nomes das espécies ────────────────────────────────────────────────────────
BIOFLORE_SPECIES = {
    1: "Pterodon\nemarginatus",
    2: "Qualea\nparviflora",
    3: "Salvertia\nconvallariodora",
    4: "Tachigali\naurea",
}

AMAZON_SPECIES = {
    1:  "Abiorana\nRosa",
    2:  "Angico\nAngico",
    3:  "Angico\nVermelho",
    4:  "Castanheira",
    5:  "Cedro",
    6:  "Cerejeira",
    7:  "Cumaru\nFerro",
    8:  "Garapeira",
    9:  "Guaribeiro",
    10: "Guariuba",
    11: "Ipê",
    12: "Jutaí",
    13: "Massaranduba",
    14: "Samaúma",
    15: "Tachi",
    16: "Tauari",
    17: "Ucuuba",
}

# ── Carrega F1 por espécie ────────────────────────────────────────────────────

def load_bioflore_species_f1(versions, base_dir):
    """
    Retorna dict {label: {"mean": array, "std": array}} com F1 por classe,
    lendo global_component_metrics.yaml de cada iteração.
    """
    result = {}
    for label, folder, avg_iters in versions:
        version_path = base_dir / folder
        with open(version_path / "args.yaml") as f:num_iter = yaml.safe_load(f).get("num_iter", 0)

        all_f1 = []
        for i in range(1, num_iter + 1):
            p = version_path / f"iter_{i:03d}" / "global_component_metrics.yaml"
            if p.exists():
                with open(p) as fh:
                    m = yaml.safe_load(fh)
                f1 = m.get("global/component/F1")
                if f1:
                    all_f1.append(f1)

        if not all_f1:
            continue

        arr = np.array(all_f1)          # shape (n_iters, n_classes)
        if avg_iters:
            result[label] = {"mean": arr.mean(axis=0), "std": arr.std(axis=0)}
        else:
            result[label] = {"mean": arr[0], "std": np.zeros(arr.shape[1])}

    return result


def load_amazon_species_f1(versions, base_dir):
    """
    Retorna dict {label: {"mean": array, "std": array}} apenas para métodos
    que possuem F1 por espécie (all_labels_F1_component) armazenado.
    Métodos sem esse dado são simplesmente omitidos.
    """
    data = {}

    for label, folder, avg_iters in versions:
        version_path = base_dir / folder
        with open(version_path / "args.yaml") as f:
            num_iter = yaml.safe_load(f).get("num_iter", 0)

        per_species = []
        for i in range(1, num_iter + 1):
            c = version_path / f"iter_{i:03d}" / "all_labels_test_metrics.yaml"
            if not c.exists():
                continue
            with open(c) as fh:
                m = yaml.safe_load(fh)
            f1_list = m.get("all_labels_F1_component")
            if f1_list:
                per_species.append(f1_list)

        if not per_species:
            continue  # método sem dados por espécie — omitir

        arr = np.array(per_species)
        if avg_iters:
            data[label] = {"mean": arr.mean(axis=0), "std": arr.std(axis=0)}
        else:
            data[label] = {"mean": arr[0], "std": np.zeros(arr.shape[1])}

    return data


bioflore_species_data = load_bioflore_species_f1(
    BIOFLORE_VERSIONS, REPO_ROOT / "bioflore_data"
)
amazon_species_data = load_amazon_species_f1(
    AMAZON_VERSIONS, REPO_ROOT / "amazon_data"
)

print("Bioflore — métodos com dados por espécie:", list(bioflore_species_data.keys()))
print("Amazon  — métodos com dados por espécie:", list(amazon_species_data.keys()))


In [ ]:
def plot_species_bars(ax, data, species_dict, colors=COLORS, show_std=False, label_rotation=0):
    """
    Grouped bar chart (species x methods), paper-ready style.

    Parameters
    ----------
    ax             : matplotlib Axes
    data           : dict {method: {"mean": array, "std": array}}
    species_dict   : dict {int_class: str_name}
    colors         : colorblind-safe palette
    show_std       : whether to draw std-dev error bars
    label_rotation : x-tick label rotation in degrees (0 = horizontal)
    """
    species_labels = [species_dict[k] for k in sorted(species_dict)]
    if label_rotation != 0:
        species_labels = [lbl.replace("\n", " ") for lbl in species_labels]
    n_species = len(species_dict)
    n_methods = len(data)

    x     = np.arange(n_species)
    width = 0.72 / max(n_methods, 1)

    for idx, (method, vals) in enumerate(data.items()):
        offset = (idx - (n_methods - 1) / 2) * width
        means  = np.array(vals["mean"])
        stds   = np.array(vals["std"]) if show_std else None

        ax.bar(
            x + offset, means,
            width=width * 0.88,
            label=method,
            color=colors[method],
            yerr=stds if show_std else None,
            capsize=3,
            error_kw={"elinewidth": 1.2, "alpha": 0.8},
            alpha=1.0,
            zorder=3,
        )

    ax.set_xticks(x)
    ha = "right" if label_rotation != 0 else "center"
    ax.set_xticklabels(species_labels, fontsize=11, ha=ha, rotation=label_rotation)
    ax.set_ylabel("Component-level F1-score (%)", fontsize=12)
    ax.set_ylim(0, 112)
    ax.yaxis.set_tick_params(labelsize=11)
    ax.legend(fontsize=11, loc="lower center", bbox_to_anchor=(0.5, 1.01),
              ncol=n_methods, frameon=False)
    ax.grid(axis="y", alpha=0.25, linewidth=0.7, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(axis="x", length=0)


fig_dir = REPO_ROOT / "exploration_notebooks" / "figures" / "compare_models_metrics"
fig_dir.mkdir(parents=True, exist_ok=True)

# ── Bioflore ──────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(8, 5))
plot_species_bars(ax1, bioflore_species_data, BIOFLORE_SPECIES)
fig1.tight_layout(pad=1.5)
fig1.savefig(fig_dir / "bioflore_species_f1.pdf", dpi=300, bbox_inches="tight")
print(f"Saved: {fig_dir / 'bioflore_species_f1.pdf'}")
plt.show()


In [ ]:

# ── Amazon ────────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 5))
plot_species_bars(ax2, amazon_species_data, AMAZON_SPECIES, label_rotation=45)
fig2.tight_layout(pad=1.5)
fig2.savefig(fig_dir / "amazon_species_f1.pdf", dpi=300, bbox_inches="tight")
print(f"Saved: {fig_dir / 'amazon_species_f1.pdf'}")
plt.show()


In [ ]:
!jupyter nbconvert --to html "$REPO_ROOT/exploration_notebooks/compare_models_metrics.ipynb" --output-dir "$REPO_ROOT/exploration_notebooks"